In [26]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [27]:
texts = [
    ('../Data/True - True.csv',1),
    ('../Data/Fake - Fake.csv',0)
]

dfs = []

for file, label in texts:
    df_temp = pd.read_csv(file)
    df_temp['label'] = label
    dfs.append(df_temp)

df = pd.concat(dfs, axis=0)

df = df.sample(frac=1).reset_index(drop=True)

df.head()

,title,text,subject,date,label
0,U.S. House Speaker Ryan seeks tax bill vote by...,WASHINGTON (Reuters) - U.S. House Speaker Paul...,politicsNews,"October 24, 2017",1
1,President Curtsy Delivers A Really Sick Burn T...,Conservatives have suggested repeatedly that i...,News,"May 23, 2017",0
2,Russia says U.S. providing cover for Islamic S...,MOSCOW (Reuters) - The United States is provid...,worldnews,"November 14, 2017",1
3,Eric Trump Tries To Defend Donald On Tax Evas...,The topic of Donald Trump s taxes or rather ...,News,"October 5, 2016",0
4,Request to Halt Construction of DAPL Declined,21st Century Wire says A request by Native Ame...,US_News,"February 13, 2017",0


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44265 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [29]:
def preprocess_text(text):

    text = re.sub(r'http\S+', '', text)
    

    text = re.sub(r'[^a-zA-Z\s]', '', text)
    

    text = text.lower()


    text = text.split()
    
    return ' '.join(text)

In [30]:
df_x=df["title"].fillna('') + " " + df["text"].fillna('')
df_y=df["label"]

In [31]:
x_train, x_test, y_train, y_test = train_test_split(df_x, df_y, test_size=0.2, random_state=42)

In [32]:
x_train = x_train.apply(preprocess_text)
x_test = x_test.apply(preprocess_text)

In [33]:
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(x_train)

In [34]:
x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

In [35]:
x_train = pad_sequences(x_train, maxlen=200)
x_test = pad_sequences(x_test, maxlen=200)

In [36]:
early_stop = EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True)

In [37]:
model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=128, input_length=200))
model.add(LSTM(128))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

c:\Users\Hend\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [38]:
model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 93s 100ms/step - accuracy: 0.9451 - loss: 0.1544 - val_accuracy: 0.9660 - val_loss: 0.0954
Epoch 2/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 90s 100ms/step - accuracy: 0.9790 - loss: 0.0618 - val_accuracy: 0.9858 - val_loss: 0.0504
Epoch 3/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 95s 106ms/step - accuracy: 0.9850 - loss: 0.0451 - val_accuracy: 0.9832 - val_loss: 0.0578
Epoch 4/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 101s 112ms/step - accuracy: 0.9837 - loss: 0.0480 - val_accuracy: 0.9843 - val_loss: 0.0538


In [39]:
model.evaluate(x_test, y_test)

281/281 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.9872 - loss: 0.0471


[0.04712676629424095, 0.987193763256073]